# `pt.layout_diagram` — see what plotlet did

When a chart looks wrong — wrong panel size, fat margins, panels that should be joined but aren't, gaps that should be smaller — guess-and-check (matplotlib's default debugging mode) wastes a lot of time. plotlet ships a built-in alternative: `pt.layout_diagram(chart)` returns a schematic showing exactly what plotlet allocated where, drawn at scale.

  * Panel bboxes drawn dashed; data areas drawn solid. The colored ring between them is the margin — its thickness is its pixel size, so left vs right vs top vs bottom imbalances are visible at a glance.
  * Gaps between panels drawn as hatched slabs labeled with their pixel size (default 20, collapses to 0 for share-pair joints).
  * Numeric annotations on every band so you can read the exact values without guessing from pixel positions.

Because `pt.layout_diagram(chart)` returns a `Chart` leaf, it participates in plotlet's composition algebra. The recommended workflow is `chart | pt.layout_diagram(chart)` — chart and its schematic side by side, in one cell:

```python
c = pt.chart(...).line(xs, ys)
c | pt.layout_diagram(c)            # chart and diagram side by side
b = c | pt.legend(c)
b | pt.layout_diagram(b)            # diagram of the composed (chart + legend)
```

The schematic is built purely by re-parsing the chart's SVG and reading its public `data-plotlet-*` attrs (introduced in 0.3.0). Nothing private from `plotlet.layout` or `plotlet.core` is touched, which means the same construction is available to any consumer that wants to write its own layout linter, AI inspector, or debug viz. See [docs/AI_ATTRS.md](../docs/AI_ATTRS.md) for the schema, and [src/plotlet/layout_diagram.py](../src/plotlet/layout_diagram.py) for the implementation as a worked example.


In [23]:
import plotlet as pt
import math

xs  = [i * 0.1 for i in range(64)]
sin = [math.sin(t) for t in xs]
cos = [math.cos(t) for t in xs]


## What the diagram shows

For each panel discovered in the chart's SVG, the diagram renders three things at scale:

  * **Outer dashed rectangle** — the panel bbox plotlet allocated, margins included. Read from `data-plotlet-panel-bbox`.
  * **Inner solid rectangle** — the data area where artists actually draw. Read from `data-plotlet-data-area`. The colored ring between dashed and solid is the margin; thickness encodes pixel size.
  * **Hatched slab** between adjacent bboxes — the gap. Recovered by pairwise scan over the bbox list (no extra schema attribute needed); joined share-pair joints filter out automatically because their gap is 0.

Numeric annotations are layered on top: pixel sizes of each margin band, the data area's `w × h` in the center, the resolved scale/range info below.

`pt.layout_diagram(chart)` returns a `Chart` leaf with `_leaf_kind="debug"`. It composes through the same operators as any other chart — `|`, `/`, `pt.grid`, share_x rules don't apply to it, and parents grow to fit it just like any data leaf. Standalone calls (`pt.layout_diagram(c)`) still work for cells where you want only the schematic.


## Side-by-side: `c | pt.layout_diagram(c)`

The recommended workflow. Because the diagram is a Chart leaf, it composes with `|` and friends — render the chart and its schematic together, in one cell, in the same SVG. No separate-cell juggling and (importantly) the two are in the same `<svg>` so the visual proportions can be trusted as one-to-one.


In [24]:
c = pt.chart(title='single', data_width=300, data_height=180, xlabel='x', ylabel='sin x')
c.line(xs[:32], sin[:32])
c | pt.layout_diagram(c)


## Examples — by complexity


### 1. Single plot

This section uses standalone `pt.layout_diagram(c)` (one cell per render) to focus on the schematic itself. The side-by-side `chart | pt.layout_diagram(chart)` form is shown above; either works and both are common. Below: the diagram is the same outer size as the chart, but strips chart content so only the layout decisions are visible.


In [25]:
single = pt.chart(title='single', data_width=400, data_height=240, xlabel='x', ylabel='sin x')
single.line(xs, sin)
single


In [26]:
pt.layout_diagram(single)

**How margins behave with panel size.** Body-first leaves (set via `pt.chart(data_width=…, data_height=…)`, the default since 0.2.0) compute each side as `max(spec floor, content-required)`. Margins don't scale with panel size — they just float above the floor when the title or labels would overflow. Two more sizes show the margin staying constant while the data area changes:

> To target a specific final SVG canvas size, chain `.fit(canvas_width=…, canvas_height=…)` after composing — it rescales each leaf's data region while keeping margins, fonts, and gaps at their absolute pixel sizes.

In [27]:
c = pt.chart(title='600 × 400 (spec default)', data_width=600, data_height=400, xlabel='x', ylabel='sin x')
c.line(xs, sin)
c


In [28]:
pt.layout_diagram(c)

In [29]:
c = pt.chart(title='150 × 100', data_width=150, data_height=100, xlabel='x', ylabel='sin x')
c.line(xs, sin)
c


In [30]:
pt.layout_diagram(c)

Reading the three diagrams side by side:

  - **All three have the same margins** — L58 T30 R18 B48 — visible as identical-thickness colored rings around the data areas. Body-first margins don't scale with panel size; they're `max(floor, content-required)` and the floors win for typical content.
  - **Only the data areas differ** — 600×400 → 400×240 → 150×100 — visible as the shrinking solid inner rectangle. The bbox shrinks to match.
  - At 150×100 the panel is mostly margin: tiny solid strip, fat colored ring. This is the body-first contract: **the data region you ask for is the data region you get**, and the canvas grows around it.

Try rerendering with `title='a much much longer plot title that overflows'` — once the title is wider than the floor + base width, the top margin grows past 30 and you'll see the top band thicken.

### 2. Two-plot composition, no share

`(a | b) / c` with no `share_x=` / `share_y=`. The white band between two panels = the gap; the colored band inside each panel = its margin. `inner_gap` doesn't enter the picture — every panel keeps its full margin.


In [31]:
a = pt.chart(title='a', data_width=290, data_height=140); a.line(xs, sin)
b = pt.chart(title='b', data_width=290, data_height=140); b.line(xs, cos)
c = pt.chart(title='c', data_width=600, data_height=140)
c.line(xs, [s + co for s, co in zip(sin, cos)])
two_plot = (a | b) / c
two_plot


In [32]:
pt.layout_diagram(two_plot)

### 3. Share-pair

Auto-zero-gap only fires for *sibling leaves* in the same parent. In `(a / c).share_x() | b`, `a` and `c` are siblings in a vertical and the parent-level `.share_x()` collapses the gap between them. Both panels' inner margins on the joint shrink to `inner_gap`. `b` sits in its own column and is unaffected.


In [33]:
a = pt.chart(title='a', data_width=290, data_height=140); a.line(xs, sin)
c = pt.chart(title='c', data_width=290, data_height=140)
c.line(xs, [s + co for s, co in zip(sin, cos)])
b = pt.chart(title='b', data_width=290, data_height=280); b.line(xs, cos)
share_pair = (a / c).share_x() | b
share_pair


In [34]:
pt.layout_diagram(share_pair)

### 4. Multi-share grid (annotated-heatmap shape)

`share_x="col"` (collapses panels in same grid column) and `share_y="row"` (collapses horizontally). `main` ends up with `inner_gap` on its top *and* left edges; `tree` gets it on its right; `top` gets it on its bottom.


In [35]:
main = pt.chart(title='main', data_width=440, data_height=260)
main.line([1, 2, 3, 4, 5], [2, 4, 1, 5, 3])
top  = pt.chart(title='top',  data_width=440, data_height=80)
top.line([1, 2, 3, 4, 5], [1, 1, 3, 1, 1])
tree = pt.chart(title='tree', data_width=120, data_height=260)
tree.line([0, 1, 2], [2, 3, 4])
heat_grid = pt.grid([
    [None, top ],
    [tree, main],
], share_x="col", share_y="row")
heat_grid


In [36]:
pt.layout_diagram(heat_grid)

## Tweaking layout constants

The `inner_gap` and `gap` constants don't have a public setter yet, but `plotlet.layout` reads them as plain module-level names (`_INNER_GAP`, `_GAP`) on each call, so you can rebind them at runtime to experiment. (Restore originals at the end if you don't want global side-effects in the kernel.)

The diagram is what makes a knob change *visible*: rebind the constant, render the same shape, compare the two diagrams. You can see directly which gaps shrank, which margin sides shifted, and which panels redistributed — the kind of feedback loop that's tedious to extract from numeric prints. This section is the one piece of the notebook that still reaches into plotlet's private surface.


### Basic patch

Same `(a / c) | b` shape from §3 but with patched values — compare against §3's diagram to see the joint tighten and the b-side gap shrink.

In [37]:
import plotlet.layout as _l

def demo():
    a = pt.chart(title='a', data_width=290, data_height=140); a.line(xs, sin)
    c = pt.chart(title='c', data_width=290, data_height=140)
    c.line(xs, [s + co for s, co in zip(sin, cos)])
    b = pt.chart(title='b', data_width=290, data_height=280); b.line(xs, cos)
    return (a / c).share_x() | b

orig_inner, orig_gap = _l._INNER_GAP, _l._GAP
try:
    _l._INNER_GAP = 5   # tighter share-pair joint
    _l._GAP       = 100 # wider default spacing between non-shared panels
    patched = demo()
finally:
    _l._INNER_GAP, _l._GAP = orig_inner, orig_gap
patched


In [38]:
pt.layout_diagram(patched)

### On the grid example

Same patch applied to §4's annotated-heatmap shape.

In [39]:
orig_inner, orig_gap = _l._INNER_GAP, _l._GAP
try:
    _l._INNER_GAP = 0
    _l._GAP       = 8
    main = pt.chart(title='main', data_width=440, data_height=260)
    main.line([1, 2, 3, 4, 5], [2, 4, 1, 5, 3])
    top  = pt.chart(title='top',  data_width=440, data_height=80)
    top.line([1, 2, 3, 4, 5], [1, 1, 3, 1, 1])
    tree = pt.chart(title='tree', data_width=120, data_height=260)
    tree.line([0, 1, 2], [2, 3, 4])
    grid_patched = pt.grid([
        [None, top ],
        [tree, main],
    ], share_x="col", share_y="row")
finally:
    _l._INNER_GAP, _l._GAP = orig_inner, orig_gap
grid_patched


In [40]:
pt.layout_diagram(grid_patched)

### Isolating `_GAP` (no-share layout)

A no-share layout has no joined joints, so `_INNER_GAP` doesn't enter the picture. Tweak `_GAP` here to see the inter-panel white slabs in isolation.

In [41]:
orig_inner, orig_gap = _l._INNER_GAP, _l._GAP
try:
    _l._GAP = 80   # try 0, 4, 20, 40 to see the effect
    a = pt.chart(title='a', data_width=290, data_height=140); a.line(xs, sin)
    b = pt.chart(title='b', data_width=290, data_height=140); b.line(xs, cos)
    c = pt.chart(title='c', data_width=600, data_height=140)
    c.line(xs, [s + co for s, co in zip(sin, cos)])
    isolated = (a | b) / c
finally:
    _l._INNER_GAP, _l._GAP = orig_inner, orig_gap
isolated


In [42]:
pt.layout_diagram(isolated)

## Three-way sharing through a common reference

`A | B | C` with all three sharing y via `(A | B | C).share_y()` — three panels meant to align on a common y axis. Two questions about correctness:

1. **Do all three panels render with the same data area size?** They should: all leaves were created with `data_width=200, data_height=140`.
2. **Do all adjacent pairs join cleanly (gap=0)?** They should: A, B, C are all in the same y-share-equivalence class.

Both used to fail. Two distinct bugs:

* `_pair_gap` only checked *direct* share-pair relationships within sibling pairs, so B↔C kept the default 20 px gap even though both share with A. Fixed by walking the share-equivalence class.
* `_compute_measured_margins` set `M_eff = max(floor, content)` *before* hide_* collapse, so `canvas_w = data + M_eff` was bigger than needed and `iw = canvas - M_eff_collapsed` ran *wider* than `data_width` for joined panels. Fixed by folding hide_* into M_eff during the pre-pass.

The diagrams below now show the converged correct behavior — all data widths equal 200, all adjacent pairs joined. The same code with the same shape via `(A | B | C).share_y()` produces an identical layout.


In [43]:
# Horizontal: A | B | C with B and C both sharing y with A
A = pt.chart(title='A', data_width=200, data_height=140); A.line([0,1,2,3,4],[0,1,2,1,0])
B = pt.chart(title='B', data_width=200, data_height=140); B.line([0,1,2,3,4],[0,2,1,3,2])
C = pt.chart(title='C', data_width=200, data_height=140); C.line([0,1,2,3,4],[1,1,1,1,1])
horiz = (A | B | C).share_y()
horiz / pt.layout_diagram(horiz)


In [44]:
# Vertical: A / B / C with B and C both sharing x with A
A2 = pt.chart(title='A', data_width=200, data_height=140); A2.line([0,1,2,3,4],[0,1,2,1,0])
B2 = pt.chart(title='B', data_width=200, data_height=140); B2.line([0,1,2,3,4],[0,2,1,3,2])
C2 = pt.chart(title='C', data_width=200, data_height=140); C2.line([0,1,2,3,4],[1,1,1,1,1])
vert = (A2 / B2 / C2).share_x()
vert | pt.layout_diagram(vert)
